<a href="https://colab.research.google.com/github/steadhac/ad-image-evaluator/blob/main/outpainting_eval_synthetic_videos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Auto-Outpainting Video Evaluation Pipeline

## Goal

When a video is auto-outpainted — meaning an AI model extends its frame beyond the original edges — the result needs to be reviewed for quality before it goes live. Manual review at scale is too slow and inconsistent. This project builds an **automated evaluation pipeline** that ingests a pair of videos (original + outpainted asset), runs a series of structured quality checks, and produces a verdict for each check that matches what a trained human rater would give.

The pipeline is designed to be used alongside a **golden dataset** — a small set of hand-rated video pairs that serve as the ground truth. Once the pipeline is built and graded against the golden set, it can be deployed to auto-approve high-confidence passes, flag borderline cases for human review, and reject clear failures — reducing manual review load while keeping quality bar consistent.

---

## The 8 phases

| Phase | Name | What it builds |
|---|---|---|
| **0** | Golden set + routing spec | Hand-rated example pairs in a structured spreadsheet; defines what each check looks like |
| **1** | Ingestion + Gate 1 | Verifies both videos in a pair can be opened and decoded before anything else runs |
| **2** | Frame extraction | Samples key frames from both videos and crops the seam boundary region |
| **3** | Context fetch | Loads metadata (category, guideline version) needed for content-aware checks |
| **4** | Per-question routing | Routes each of the 7 quality questions to the cheapest viable mechanism: rule, optical flow, or VLM prompt |
| **5** | Output formatting | Shapes the results into the exact column structure of the golden spreadsheet |
| **6** | Grading | Compares pipeline output against the golden set and measures agreement rate per question |
| **7** | Selective shipping | Auto-approves high-confidence correct answers; flags the rest for human review |
| **8** | Monitoring | Re-runs the golden set periodically to detect model or quality drift over time |

---

## Design principles

**Cheapest viable mechanism first.** Rules run before classifiers; classifiers run before VLM prompts. A VLM is only invoked when a cheaper check can't reliably answer the question.

**Gates stop the pair early.** If a video can't be opened (Gate 1) or contains a guideline violation (Gate 2), the pair is rejected immediately and no further checks run — saving compute and avoiding meaningless results on broken input.

**Output matches the golden shape exactly.** Every field the pipeline produces maps 1:1 to a column in the golden spreadsheet, making grading and comparison straightforward.

**Built incrementally.** Each phase is a standalone module with its own test cases. Phases are added one at a time and verified before the next one is built.

---

## Quality checks (7 questions + 2 gates)

| ID | Check | Type | Mechanism |
|---|---|---|---|
| Gate 1 | Video load / integrity | Gate | Rule (cv2) |
| Gate 2 | Guideline violation (content categories) | Gate | Classifier + VLM fallback |
| Q1 | Subject boundary integrity | VLM | Are human/subject boundaries intact at the seam? |
| Q2 | Object plausibility | VLM | Does the generated object look physically realistic? |
| Q3 | Style consistency | VLM | Does the generated region match the visual style of the original? |
| Q4 | Motion coherence | Rule | Optical flow in outpainted strip; prompt if borderline |
| Q5 | Compositional focus | VLM | Does the generated region distract from the focal subject? |
| Q6 | Seam visibility | Rule | Pixel diff at seam boundary |
| Q7 | Clarity consistency | VLM | Does sharpness/noise/quality match between original and generated? |

# Step 1 — Gate 1: Video Load & Integrity Check

**File:** `step1_gate.py` | **Part of:** Auto-Outpainting Video Evaluation Pipeline | **Status:** ✅ Complete

---

## What it does

The first gate in the pipeline. Before any quality evaluation runs, both videos in a pair must be readable. This step checks that each file can be opened, has at least one frame, and the first frame can be decoded. If either file fails, the pair is stopped here — no downstream checks run.

## Interface

**Input:** `gate1_load(original_path, asset_path)` — two file paths

**Output:** dict with three keys:

| key | values |
|---|---|
| `status` | `"ok"` or `"failed"` |
| `failed_side` | `null`, `"original"`, `"asset"`, or `"both"` |
| `reason` | human-readable string |

## Quality checks (7 questions + 2 gates)

| ID | Question | Mechanism | Prompt? |
|---|---|---|---|
| Gate 1 | Video load / integrity | Rule (cv2) | No |
| Gate 2 | Guideline violation (content categories) | Classifier + VLM fallback | Fallback only |
| Q1 | Subject boundary integrity | VLM | Yes |
| Q2 | Object plausibility | VLM | Yes |
| Q3 | Style consistency | VLM | Yes |
| Q4 | Motion coherence | Rule — optical flow in outpainted strip | Fallback only |
| Q5 | Compositional focus | VLM | Yes |
| Q6 | Seam visibility | Rule — pixel diff at seam boundary | No |
| Q7 | Clarity consistency | VLM | Yes |

## Test cases (synthetic videos in `eval_videos/`)

| pair | original | asset | expected |
|---|---|---|---|
| 1 | valid mp4 | valid mp4 | `status: ok` |
| 3 | valid mp4 | corrupt bytes | `status: failed, failed_side: asset` |
| — | missing | missing | `status: failed, failed_side: both` |

In [3]:
!cat /content/drive/MyDrive/outpainting_eval_pipeline/step1_gate.py

import cv2
from typing import Optional

def check_video(path):
    cap = cv2.VideoCapture(path)
    if not cap.isOpened():
        return False, f"cv2 could not open: {path}"
    fc = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if fc == 0:
        cap.release()
        return False, f"0 frames: {path}"
    ret, frame = cap.read()
    cap.release()
    if not ret or frame is None:
        return False, f"first frame not decodable: {path}"
    return True, "ok"

def gate1_load(original_path, asset_path):
    orig_ok, orig_r = check_video(original_path)
    asset_ok, asset_r = check_video(asset_path)
    if orig_ok and asset_ok:
        return {"status":"ok","failed_side":None,"reason":"both videos loaded OK"}
    if not orig_ok and not asset_ok:
        return {"status":"failed","failed_side":"both","reason":f"orig: {orig_r} | asset: {asset_r}"}
    if not orig_ok:
        return {"status":"failed","failed_side":"original","reason":orig_r}
    return {"status":"failed","failed_side":"asse

In [4]:
!python3 /content/drive/MyDrive/outpainting_eval_pipeline/step1_gate.py

[PASS] Pair1 both valid
  {"status": "ok", "failed_side": null, "reason": "both videos loaded OK"}
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x2e800c0] moov atom not found
[PASS] Pair3 corrupt asset
  {"status": "failed", "failed_side": "asset", "reason": "cv2 could not open: /content/drive/MyDrive/eval_videos/pair3_outpainted_CORRUPT.mp4"}
[PASS] Both missing
  {"status": "failed", "failed_side": "both", "reason": "orig: cv2 could not open: /tmp/nope_orig.mp4 | asset: cv2 could not open: /tmp/nope_asset.mp4"}

ALL PASSED


# Step 2 — Frame Extraction

**File:** `step2_frames.py` | **Part of:** Auto-Outpainting Video Evaluation Pipeline | **Status:** ✅ Complete

---

## What it does

Takes a video pair that has already passed Gate 1 and extracts a small, representative set of image frames from both videos. It also cuts a narrow vertical strip of pixels right at the boundary between the original and generated regions — this "seam crop" is what Q6 (Seam visibility) will score later.

## Why sample frames instead of using all of them?

A typical video has 24–30 frames per second. Extracting every frame would be slow and mostly redundant — consecutive frames look nearly identical. Instead we take 7 frames that cover the full timeline:

- **Frame 0** — how the video opens
- **Frame N-1** — how the video closes
- **5 evenly-spaced interior frames** — motion across the middle

## Interface

**Input:** `extract_frames(orig_path, asset_path, n_uniform=5, seam_half=20)`

**Output:** dict with five keys:

| key | description |
|---|---|
| `frames` | list of `(frame_idx, orig_np, asset_np)` — paired BGR frames at each sampled moment |
| `seam_crops` | list of `(frame_idx, seam_np)` — 40px strip around the seam boundary, from the asset frame |
| `seam_x` | x-coordinate of the seam = original video width in pixels |
| `status` | `"ok"` or `"error"` |
| `message` | empty on success, error detail on failure |

## Test cases

| name | orig | asset | expected |
|---|---|---|---|
| pair1 — both valid | pair1_original.mp4 | pair1_outpainted.mp4 | `status: ok, 7 frames, 7 seam crops, seam_x=640` |
| missing asset | pair1_original.mp4 | nonexistent.mp4 | `status: error` |
| pair3 — asset missing | pair3_original.mp4 | nonexistent.mp4 | `status: error` |

In [5]:
!cat /content/drive/MyDrive/outpainting_eval_pipeline/step2_frames.py

"""
step2_frames.py  --  Frame Extraction Module

PURPOSE
-------
Gate 1 only tells us the videos can be opened and are roughly the same length.
This module does the actual work of pulling image frames out of both videos
so that later steps can inspect the visual content.

It also cuts a narrow "seam strip" -- the vertical slice of pixels right at
the join between the original region and the outpainted region. That strip
is what Q6 (Seam visibility) will score.

WHY NOT EXTRACT ALL FRAMES?
---------------------------
A typical video has 24-30 frames per second. Extracting every frame would be
slow and wasteful -- most consecutive frames look nearly identical. Instead we
sample a small, representative set:
  - Frame 0    : the very first frame (how the video opens)
  - Frame N-1  : the very last frame  (how the video closes)
  - 5 interior frames spaced evenly across the middle
This gives 7 frames total and covers motion across the full timeline.

INPUTS
------
orig_path  : str   path t

In [6]:
!python3 /content/drive/MyDrive/outpainting_eval_pipeline/step2_frames.py

[PASS] pair1 -- both videos valid
       status=ok | 7 frames, 7 seam crops, seam_x=640
[PASS] missing asset file
       status=error | Could not open one or both videos
[PASS] pair3 -- orig valid but asset missing
       status=error | Could not open one or both videos

Result: 3/3 tests passed
